# SFCmodel

> The modeler class which will run equations and store data.

In [ ]:
#| default_exp SFCmodel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import warnings
import numpy as np
import pandas as pd
import sympy as sp
from scipy.optimize import root
from tqdm.auto import tqdm

from SFCmodeler.term import Term, Variable, Parameter
from SFCmodeler.modelresults import ModelResults

The SFCmodel class holds the model equations and solver.

In [ ]:
#| export

class SFCmodel:
    def __init__(self, equations=None, redundant_equation=None,start_period="2000",freq="Y"):
        self.equations = equations or []
        self.terms = sorted(list({sym for eq in self.equations for sym in eq.free_symbols}), key=str)
        self.lags = [var for var in self.terms if var.lag != 0]
        self.parameters = [var for var in self.terms if isinstance(var, Parameter) and var.lag == 0]

        for term in list(self.lags):
          # capture only-lagged parameters
          if term.unlagged not in self.terms: self.terms.append(term.unlagged)
          if isinstance(term, Parameter) and term.unlagged not in self.parameters: self.parameters.append(term.unlagged)
          # make sure Y(-1) ... Y(-(k-1)) exist too
          for j in range(1, -term.lag):
            mid = term.unlagged(-j)
            if mid not in self.terms: self.terms.append(mid)
            if mid not in self.lags: self.lags.append(mid)

        self.variables = [var for var in self.terms if isinstance(var, Variable) and var not in self.lags]
        self.endogenous = [eq.lhs for eq in self.equations]

        if set(self.variables) != set(self.endogenous):
          raise ValueError(f"Variables and equations do not match. Variables are {self.variables}, equations are {self.endogenous}")
        if len(set(self.endogenous)) != len(self.endogenous):
          raise ValueError("Each variable can only appear on the LHS of one equation.")

        self.redundant_equation = redundant_equation

        self.data = ModelResults(pd.DataFrame(
            {term: term.initial_value for term in self.terms},
            index=pd.period_range(start_period, periods=1, freq=freq), dtype=float
        ))



    def __setitem__(self, key, value):
        if isinstance(key, Term):
            items = {key: value}
        elif isinstance(key, slice) and key.start is None and key.stop is None:
            items = dict(value)
        else:
            raise TypeError(f"Key must be a Term or [:], not {type(key).__name__}")

        unknown = [k for k in items if k not in self.data.df.columns]
        if unknown:
            raise KeyError(f"Not in this model: {unknown}")


        if len(self.data.df) > 1:
          # doing a shock, rather than editing initial conditions
          self.carry_forward()

        lagged = [k for k in items if k.lag != 0]
        if lagged and len(self.data.df) > 1:
            warnings.warn(
                f"Overriding lagged term(s) {lagged} after the first period; this breaks "
                "the link to last period's values. Set the earlier period's value instead "
                "if you want stock-flow consistency.",
                stacklevel=2,
            )

        self.data.df.loc[self.data.df.index[-1], list(items)] = list(items.values())


    def __getitem__(self, key):
      if not isinstance(key, Term):
          raise TypeError(f"Key must be a Term, not {type(key)}")

      col = self.data.df[key]
      prev = col.iloc[-2] if len(col) > 1 else pd.NA
      for val in (col.iloc[-1], prev):
          if not pd.isna(val):
              return float(val)
      return float("nan")

    
    def equation_values(self, variable):
        "Returns the data for all the values in the equation for a specified variable."
        eqn = [eqn for eqn in model.equations if eqn.lhs == variable][0]
        try:
            from IPython.display import display
        except ImportError:
            display = print
        display(eqn)
        return model.data[[variable] + sorted(list(eqn.rhs.free_symbols),key=str)]
    

    def carry_forward(self):
        if self.data.df.iloc[-1].isna().any():
          return  # Model hasn't been solved yet

        if len(self.data.df) == 1:
            self.data.df = self.data.df.astype(float)

        new_row = {term: np.nan for term in self.endogenous}

        for param in self.parameters:
            new_row[param] = self.data.df[param].iloc[-1]

        # History shifts by one period: Y(-1) <- Y, Y(-2) <- Y(-1), ...
        last = self.data.df.iloc[-1]
        for lag in self.lags:
            new_row[lag] = last[lag.unlagged(lag.lag + 1)]

        # Append new row to DataFrame
        next_idx = self.data.index[-1] + 1
        new_df_row = pd.DataFrame([new_row], index=[next_idx])
        self.data.df = pd.concat([self.data.df, new_df_row])



    def solve(self, periods=1, steady=False, redundant_tol=1e-6, **kwargs):
        with tqdm(total=periods) as bar:
          bar.set_description("Compiling")

          residuals = [eq.lhs - eq.rhs for eq in self.equations]
          args = self.endogenous + self.lags + self.parameters
          raw_func = sp.lambdify(args, residuals, modules="numpy")
          raw_jac = sp.lambdify(args, sp.Matrix(residuals).jacobian(self.endogenous), modules="numpy",cse=True)

          if self.redundant_equation is not None:
            redundant = self.redundant_equation
            redundant_syms = list(redundant.free_symbols)
            redundant_func = sp.lambdify(redundant_syms, [redundant.lhs, redundant.rhs], modules="numpy")

          bar.set_description("Solving")

          for t in range(periods):
            self.carry_forward()

            if len(self.data.df) == 1:
              missing = list(self.data.df.columns[self.data.df.iloc[-1].isna().to_numpy()])
              raise ValueError(
                  f"Initial values missing for {missing}. "
                  "Set them with model[term] = value, or change Term.default_value.")


            known_vals = self.data.df.iloc[-1][self.lags + self.parameters].to_numpy(dtype=float)

            # Wrap function to flatten output from shape (N, 1) to (N,)
            def residual_func(x):
                return np.array(raw_func(*x, *known_vals), dtype=float).flatten()
            def jac_func(x):
                return np.array(raw_jac(*x, *known_vals), dtype=float)

            # Initial guess
            x0 = self.data.df[self.endogenous].iloc[-2].values.astype(float)

            solution = root(residual_func, x0, jac=jac_func,**kwargs)
            if not solution.success:
              solution = root(residual_func, x0,**kwargs) # sometimes solves better without jacobian
            if not solution.success:
              raise RuntimeError(f"Solver failed for period {self.data.df.index[-1]}: {solution.message}")

            # success
            self.data.df.loc[self.data.df.index[-1], self.endogenous] = solution.x

            if self.redundant_equation is not None:
              lhs, rhs = redundant_func(*[float(self.data.df[s].iloc[-1]) for s in redundant_syms])
              if abs(lhs - rhs) > redundant_tol * max(abs(lhs), abs(rhs)):
                warnings.warn(
                    f"Redundant equation violated in period {self.data.df.index[-1]}: "
                    f"lhs = {lhs:.6g}, rhs = {rhs:.6g}",
                    stacklevel=2,
                )


            if steady:
                if self.check_steady_state(**(steady if isinstance(steady, dict) else {})):
                    bar.set_description("Reached steady state")
                    return self.data
                if t == periods - 1:
                    warnings.warn("Reached end of alloted timesteps but did not find steady-state.", stacklevel=2)
              
            bar.update(1)

        return self.data

    
    def check_steady_state(self, ignore=[], growth=[], **kwargs):
        if not "rtol" in kwargs: kwargs['rtol'] = 1e-4
        if "rtol_g" in kwargs:
            rtol_g = kwargs['rtol_g'] 
            del kwargs['rtol_g']
        else:
            rtol_g = 0
        if "atol_g" in kwargs:
            atol_g = kwargs['atol_g'] 
            del kwargs['atol_g']
        else:
            atol_g = 1e-5
        
        
        growth += [lag for lag in self.lags if lag.unlagged in growth]
        ignore += [lag for lag in self.lags if lag.unlagged in ignore]
        
        levels = self.data.df[[var for var in self.variables if var not in growth + ignore]]
        growth = self.data.df[growth].pct_change()
        
        levels = np.isclose(levels.iloc[-2],levels.iloc[-1],**kwargs).all()
        
        kwargs['rtol'] = rtol_g
        kwargs['atol'] = atol_g
        growth = np.isclose(growth.iloc[-2],growth.iloc[-1],**kwargs).all()
        
        return levels and growth

    
    def check_redundant_equation(self,redundant_tol=1e-6):
      if self.redundant_equation is not None:
          redundant = self.redundant_equation
          redundant_syms = list(redundant.free_symbols)
          redundant_func = sp.lambdify(redundant_syms, [redundant.lhs, redundant.rhs], modules="numpy")
          lhs, rhs = redundant_func(*[float(self.data.df[s].iloc[-1]) for s in redundant_syms])
          return {
              'lhs':lhs,
              'rhs':rhs,
              'inside tolerance':abs(lhs - rhs) < redundant_tol * max(abs(lhs), abs(rhs))
          }
      else:
        raise ValueError("No redundant equation defined")

First create the `Parameters` and `Variables`, and then create a model by writing equations in a list:

In [ ]:
Variable.default_value = 10

C = Variable("C","Consumption")
T = Variable("T","Taxes")
V = Variable("V","Wealth",100)
Y = Variable("Y","GDP")

alpha1 = Parameter("alpha_1","MPC out of income",0.6)
alpha2 = Parameter("alpha_2","MPC out of wealth",0.4)
G = Parameter("G","Government spending",5)
theta = Parameter("theta","Tax rate",0.2)

In [ ]:
model = SFCmodel(
    [
        C << alpha1 * (Y - T) + alpha2 * V(-1),
        Y << C + G,
        T << theta * Y,
        V << V(-1) + Y - T - C
    ]
)

In [ ]:
#| hide
assert model.terms == [C, G, T, V, V(-1), Y, alpha1, alpha2, theta]
assert model.parameters == [G, alpha1, alpha2, theta]
assert model.variables == [C, T, V, Y] # alphabetical order
assert model.endogenous == [C, Y, T, V] # equation entry order
assert model.lags == [V(-1)]

Optional additional arguments include a redundant equation, and a start and period length for the time index (which defaults to yearly begining with 2000). 

In [ ]:
model = SFCmodel(
    [
        C << alpha1 * (Y - T) + alpha2 * V(-1),
        Y << C + G,
        T << theta * Y,
        V << V(-1) + Y - T - C
    ],
    redundant_equation = V << V(-1) + G - T,
    start_period="2020",
    freq="Y"  # these are pandas PeriodIndex strings
)

In [ ]:
#| hide

assert model.redundant_equation == V << V(-1) + G - T
assert isinstance(model.data.df.index, pd.PeriodIndex)
assert model.data.df.index[0].year == 2020

If you have not already set inital_values on the `Variable`s and `Parameter`s, you can set them on the `SFCmodel`, either one at a time:

In [ ]:
model[Y] = 10

In [ ]:
#| hide
assert model.data[V(-1)].squeeze() == 100
assert model.data[V].squeeze() == 100

model[V] = 110

assert model.data[V(-1)].squeeze() == 100
assert model.data[V].squeeze() == 110
assert len(model.data) == 1

# restore
model[V] = 100

Or as a whole set using ":" and a `dict` (or anything that will parse to one):

In [ ]:
model[:] = {
 C: 10.0,
 G: 5.0,
 T: 10.0,
 V: 100.0,
 V(-1): 100.0,
 Y: 10.0,
 alpha1: 0.6,
 alpha2: 0.4,
 theta: 0.2
}

In [ ]:
#| hide
model[:] = {
 C: 100.0,
 G: 50.0,
 T: 100.0,
 V: 1000.0,
 V(-1): 1000.0,
 Y: 100.0,
 alpha1: 0.7,
 alpha2: 0.3,
 theta: 0.1
}

assert model.data.squeeze().to_dict() == {
 C: 100.0,
 G: 50.0,
 T: 100.0,
 V: 1000.0,
 V(-1): 1000.0,
 Y: 100.0,
 alpha1: 0.7,
 alpha2: 0.3,
 theta: 0.1
}

# restore
model[:] = {
 C: 10.0,
 G: 5.0,
 T: 10.0,
 V: 100.0,
 V(-1): 100.0,
 Y: 10.0,
 alpha1: 0.6,
 alpha2: 0.4,
 theta: 0.2
}

In [ ]:
# This will work too
model[:] = [
 (C, 100.0),
 (G, 50.0),
 (T, 100.0),
 (V, 1000.0),
 (V(-1), 1000.0),
 (Y, 100.0),
 (alpha1, 0.7),
 (alpha2, 0.3),
 (theta, 0.1)
]

In [ ]:
#| hide
model[:] = [
 (C, 2000.0),
 (G, 60.0),
 (T, 300.0),
 (V, 10500.0),
 (V(-1), 1001.0),
 (Y, 102.0),
 (alpha1, 0.9),
 (alpha2, 0.1),
 (theta, 0.9)
]

assert model.data.squeeze().to_dict() == {
 C: 2000.0,
 G: 60.0,
 T: 300.0,
 V: 10500.0,
 V(-1): 1001.0,
 Y: 102.0,
 alpha1: 0.9,
 alpha2: 0.1,
 theta: 0.9}

# restore
model[:] = {
 C: 10.0,
 G: 5.0,
 T: 10.0,
 V: 100.0,
 V(-1): 100.0,
 Y: 10.0,
 alpha1: 0.6,
 alpha2: 0.4,
 theta: 0.2
}

The redundant equation refers to the accounting identity that is overdetermined, as typical in SFC models. If you specify it, then the solver will check that it holds (within solver error) every period, and warn you if not. You can also check it manually using `.check_redundant_equation`:

In [ ]:
model.check_redundant_equation()

{'lhs': 100.0, 'rhs': 95.0, 'inside tolerance': False}

In [ ]:
#| hide

check = model.check_redundant_equation()

assert check['lhs'] == 100
assert check['rhs'] == 95
assert not check['inside tolerance']

In this case the check fails because our default values are not consistent (though that won't matter for this model, because the equations don't look back to the values of these variables from previous periods).

`.solve()` runs the solver for the (optional) specified number of periods, and returns the results

In [ ]:
model.solve()   # solves one period
model.solve(3)  # solves 3 more periods

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

|      |     $C$ |   $G$ |     $T$ |      $V$ |   $V_{-1}$ |     $Y$ |   $\alpha_{1}$ |   $\alpha_{2}$ |   $\theta$ |
|:-----|--------:|------:|--------:|---------:|-----------:|--------:|---------------:|---------------:|-----------:|
| 2020 | 10      |     5 | 10      | 100      |   100      | 10      |            0.6 |            0.4 |        0.2 |
| 2021 | 81.5385 |     5 | 17.3077 |  87.6923 |   100      | 86.5385 |            0.6 |            0.4 |        0.2 |
| 2022 | 72.071  |     5 | 15.4142 |  77.2781 |    87.6923 | 77.071  |            0.6 |            0.4 |        0.2 |
| 2023 | 64.0601 |     5 | 13.812  |  68.4661 |    77.2781 | 69.0601 |            0.6 |            0.4 |        0.2 |
| 2024 | 57.2816 |     5 | 12.4563 |  61.0098 |    68.4661 | 62.2816 |            0.6 |            0.4 |        0.2 |

Note that the first row is the initial conditions.

To access the same results, use `model.data`:

In [ ]:
model.data

|      |     $C$ |   $G$ |     $T$ |      $V$ |   $V_{-1}$ |     $Y$ |   $\alpha_{1}$ |   $\alpha_{2}$ |   $\theta$ |
|:-----|--------:|------:|--------:|---------:|-----------:|--------:|---------------:|---------------:|-----------:|
| 2020 | 10      |     5 | 10      | 100      |   100      | 10      |            0.6 |            0.4 |        0.2 |
| 2021 | 81.5385 |     5 | 17.3077 |  87.6923 |   100      | 86.5385 |            0.6 |            0.4 |        0.2 |
| 2022 | 72.071  |     5 | 15.4142 |  77.2781 |    87.6923 | 77.071  |            0.6 |            0.4 |        0.2 |
| 2023 | 64.0601 |     5 | 13.812  |  68.4661 |    77.2781 | 69.0601 |            0.6 |            0.4 |        0.2 |
| 2024 | 57.2816 |     5 | 12.4563 |  61.0098 |    68.4661 | 62.2816 |            0.6 |            0.4 |        0.2 |

In [ ]:
#| hide
assert isinstance(model.data, ModelResults)

`model.data` is an instance of `ModelResults`, which is a thin wrapper around a pandas `DataFrame`, but which allows for some fancy slicing and indexing. See [that page](./02_modelresults.ipynb) for more details.

One more useful way to inspect the data is with `.equation_values()`, which will return the data for all the terms in the equation for the specified variable (also as `ModelResults`)

In [ ]:
model.equation_values(C)

Eq(C, alpha_1*(-T + Y) + alpha_2*V_{-1})

|      |     $C$ |     $T$ |   $V_{-1}$ |     $Y$ |   $\alpha_{1}$ |   $\alpha_{2}$ |
|:-----|--------:|--------:|-----------:|--------:|---------------:|---------------:|
| 2020 | 10      | 10      |   100      | 10      |            0.6 |            0.4 |
| 2021 | 81.5385 | 17.3077 |   100      | 86.5385 |            0.6 |            0.4 |
| 2022 | 72.071  | 15.4142 |    87.6923 | 77.071  |            0.6 |            0.4 |
| 2023 | 64.0601 | 13.812  |    77.2781 | 69.0601 |            0.6 |            0.4 |
| 2024 | 57.2816 | 12.4563 |    68.4661 | 62.2816 |            0.6 |            0.4 |

In [ ]:
#| hide
assert model.equation_values(C).df.equals(model.data[[C,T,V(-1),Y,alpha1,alpha2]].df);

Eq(C, alpha_1*(-T + Y) + alpha_2*V_{-1})

The easiest way to shock a variable is to assign it a new value and then run the solver again:

In [ ]:
model[alpha1] = 0.7

model.solve(4)

  0%|          | 0/4 [00:00<?, ?it/s]

|      |     $C$ |   $G$ |      $T$ |      $V$ |   $V_{-1}$ |     $Y$ |   $\alpha_{1}$ |   $\alpha_{2}$ |   $\theta$ |
|:-----|--------:|------:|---------:|---------:|-----------:|--------:|---------------:|---------------:|-----------:|
| 2020 | 10      |     5 | 10       | 100      |   100      | 10      |            0.6 |            0.4 |        0.2 |
| 2021 | 81.5385 |     5 | 17.3077  |  87.6923 |   100      | 86.5385 |            0.6 |            0.4 |        0.2 |
| 2022 | 72.071  |     5 | 15.4142  |  77.2781 |    87.6923 | 77.071  |            0.6 |            0.4 |        0.2 |
| 2023 | 64.0601 |     5 | 13.812   |  68.4661 |    77.2781 | 69.0601 |            0.6 |            0.4 |        0.2 |
| 2024 | 57.2816 |     5 | 12.4563  |  61.0098 |    68.4661 | 62.2816 |            0.6 |            0.4 |        0.2 |
| 2025 | 61.8271 |     5 | 13.3654  |  52.6444 |    61.0098 | 66.8271 |            0.7 |            0.4 |        0.2 |
| 2026 | 54.2221 |     5 | 11.8444  |  45.7999 |    52.6444 | 59.2221 |            0.7 |            0.4 |        0.2 |
| 2027 | 47.9999 |     5 | 10.6     |  40.1999 |    45.7999 | 52.9999 |            0.7 |            0.4 |        0.2 |
| 2028 | 42.909  |     5 |  9.58181 |  35.6181 |    40.1999 | 47.909  |            0.7 |            0.4 |        0.2 |

In [ ]:
#| hide

data = model.data.copy()

# not steady state yet
assert not model.check_steady_state()

# run to steady state
model.solve(100)
assert model.check_steady_state()

# restore
model.data = data
data = data.copy()

# run to steady state
model.solve(100, steady=True)
assert model.check_steady_state()

# restore
model.data = data

# check that there's a warning with too few periods
with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter("always")  # Ensure all matching warnings are caught
    model.solve(2, steady= True)
    assert len(captured_warnings) == 1

# restore
model.data = data
data = data.copy()

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Instead of specifying a certain number of periods to solve, you can also ask the solver to find the **steady state** of the model, using the `steady` argument:

In [ ]:
model.solve(100, steady=True);
# The number 100 becomes give_up_steps, with the solver searching for a steady state but stopping after give_up_steps.

  0%|          | 0/100 [00:00<?, ?it/s]

Under the hood, the steady state checker calls `.check_steady_state`, which uses `np.isclose` to compare the last two timesteps. For models with randomness or growth, instead of `steady=True`, pass a `dict` that can specify a list of variables to `ignore` or a list of variables to check their `growth` rates instead of levels:

In [ ]:
# rewind the clock:
model.data = model.data[:"2028"]

model.solve(100, steady={
    'ignore':[C, T],
    'growth':[V,G]
});

  0%|          | 0/100 [00:00<?, ?it/s]

You can also include any `kwargs` to `np.isclose` in the `steady` `dict` (or in arguments to `check_steady_state()` if you want to call it manually), but specify `rol_g` and `atol_g` as `rtol` and `atol` for the growth variables.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()